# Causal Verification Pipeline — Generalized

Single-hypothesis causal verification: DAG refinement → DML estimation →
GRF heterogeneity → refutations → sensitivity → structural breaks → residual diagnostics.

**All configuration is in the first two cells. Nothing is hardcoded downstream.**

## Configuration

In [1]:
# ═══════════════════════════════════════════════════════════════
# HYPOTHESIS INPUTS — the only cell that changes between runs
# ═══════════════════════════════════════════════════════════════

# --- Data ---
DATA_PATH = './cold_chain_output/shipments.csv'
DATE_COLUMN = 'date'                          # temporal column for sorting/aggregation
ENTITY_COLUMN = 'hub_id'                      # entity for structural break aggregation
STRIP_COLUMNS = [                             # god's-eye / leakage columns to remove
    'driver_care_factor', 'logger_bias_c', 'logger_has_fault',
    'max_temp_actual_c', 'container_wall_U',
    'vehicle_reefer_kw_effective', 'vehicle_cooling_efficiency',
    'detected_flag'
]

# --- Hypothesis ---
TREATMENT = 'container_age_months'            # treatment variable
OUTCOME = 'excursion_flag'                    # outcome variable
EXPECTED_DIRECTION = 1                        # +1 or -1

# --- DAG (generator output) ---
DAG_BROAD_EDGES = [
    ('container_age_months', 'excursion_flag'),
    ('ins_type_enc', 'excursion_flag'),
    ('ambient_temp_at_dispatch_c', 'excursion_flag'),
    ('hub_id_enc', 'container_age_months'),
    ('hub_id_enc', 'excursion_flag'),
    ('hub_id_enc', 'ambient_temp_at_dispatch_c'),
    ('vehicle_reefer_kw_rated', 'excursion_flag'),
    ('month', 'excursion_flag'),
    ('month', 'container_age_months'),
    ('month', 'ambient_temp_at_dispatch_c'),
    ('ambient_temp_at_dispatch_c', 'container_age_months'),
]

# --- Derived columns (categorical encodings, temporal extractions) ---
# {new_column: (source_column, transform)} where transform is 'category_codes' or 'month'
DERIVED_COLUMNS = {
    'hub_id_enc': ('hub_id', 'category_codes'),
    'month': ('date', 'month'),
    'ins_type_enc': ('container_insulation_type', 'category_codes'),
}

# --- Enrichment join (optional: add columns from other tables) ---
# Set to None to skip
ENRICHMENT = {
    'table_path': './cold_chain_output/fleet_transitions.csv',
    'table_date_columns': ['date'],
    'join_column': 'vehicle_id',                 # column in main data
    'right_key': 'new_vehicle_id',               # column in enrichment table
    'value_column': 'new_generation',             # column to extract
    'target_name': 'vehicle_generation',          # new column name in main data
    'default_value': 1,                           # for unmatched rows
    # Additional map entries: also map retired vehicles to their generation
    'extra_mappings': [
        ('retired_vehicle_id', 'retired_generation'),
    ],
    # DAG edges to add for the enrichment variable
    'dag_edges': [
        ('vehicle_generation', 'vehicle_reefer_kw_rated'),
        ('vehicle_generation', 'container_age_months'),
        ('hub_id_enc', 'vehicle_generation'),
    ],
}

# --- GRF effect modifiers (columns whose interaction with treatment to discover) ---
GRF_EFFECT_MODIFIERS = ['ins_type_enc', 'ambient_temp_at_dispatch_c', 'vehicle_reefer_kw_rated']

# --- GRF slicing (how to group CATEs for reporting) ---
# {label: (column, method)} where method is 'unique' (categorical) or 'quartile' (continuous)
GRF_SLICES = {
    'insulation type': ('container_insulation_type', 'unique'),
    'reefer kw': ('vehicle_reefer_kw_rated', 'quartile'),
    'ambient temp': ('ambient_temp_at_dispatch_c', 'quartile'),
}

# --- Breakpoint (from Phase 2.5 SHAP scan; None if no breakpoint detected) ---
THRESHOLD_BREAKPOINT = 30

# --- Threshold sensitivity grid (centered around breakpoint) ---
THRESHOLD_OFFSETS = [-6, -3, 0, 3, 6]          # offsets from breakpoint

# --- Externalization (domain literature comparison; None to skip) ---
EXTERNALIZATION = {
    'ranking_variable': 'container_insulation_type',  # categorical to compare
    'domain_ranking': ['VIP_panel', 'PUR_foam', 'PIR_foam', 'XPS_foam'],  # slowest→fastest degradation
    'domain_ratio': 4.0,                         # expected ratio fastest/slowest from literature
    'allocation_check_column': 'hub_id',          # check allocation bias across this grouping
}

# --- Known events table for structural break matching (None to skip) ---
KNOWN_EVENTS = {
    'table_path': './cold_chain_output/fleet_transitions.csv',
    'date_column': 'date',
    'entity_column': 'hub_id',
    'match_window_months': 3,
}

# --- Measurement metadata fields (flagged in residual diagnostics) ---
MEASUREMENT_METADATA = ['logger_months_since_cal']

# --- ID columns to exclude from residual correlation scan ---
ID_COLUMNS = ['shipment_id', 'hub_id', 'hub_id_enc', 'month']

# --- Model parameters ---
LGBM_PARAMS = dict(n_estimators=300, max_depth=6, learning_rate=0.05, verbose=-1)
BOOTSTRAP_SAMPLES = 20
GRF_ESTIMATORS = 200
GRF_MIN_LEAF = 50
RANDOM_STATE = 42

# --- Quality gate thresholds ---
R2_ABORT = 0.01
R2_FLAG = 0.10
TYPICAL_EFFECT_MAX = 0.05                       # max plausible single-variable effect on outcome
MAGNITUDE_MULTIPLIER = 3                         # max_plausible = TYPICAL_EFFECT_MAX × this
VIF_THRESHOLD = 10
OVERLAP_THRESHOLD = 0.70
CC_SHIFT_THRESHOLD = 0.10
SUBSET_SHIFT_THRESHOLD = 0.20
CI_ALPHA = 0.05

## Imports

In [2]:
import pandas as pd
import numpy as np
import math
import warnings
warnings.filterwarnings('ignore')

import dowhy
from lightgbm import LGBMRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import pearsonr
import networkx as nx
from itertools import combinations
import ruptures
from econml.dml import CausalForestDML, LinearDML
from econml.inference import BootstrapInference

## Utility Functions

In [3]:
def edges_to_dot(edges):
    """Convert edge list to DOT format for DoWhy."""
    lines = ['digraph {']
    for src, dst in edges:
        lines.append(f'    {src} -> {dst};')
    lines.append('}')
    return '\n'.join(lines)


def test_conditional_independence(data, node_a, node_b, cond_set, p_threshold=CI_ALPHA):
    """Test if A ⊥ B | S in data via partial correlation."""
    a_vals = data[node_a].values.astype(float)
    b_vals = data[node_b].values.astype(float)
    if len(cond_set) == 0:
        r, p = pearsonr(a_vals, b_vals)
    else:
        X_cond = data[list(cond_set)].values.astype(float)
        lr_a = LinearRegression().fit(X_cond, a_vals)
        lr_b = LinearRegression().fit(X_cond, b_vals)
        res_a = a_vals - lr_a.predict(X_cond)
        res_b = b_vals - lr_b.predict(X_cond)
        r, p = pearsonr(res_a, res_b)
    return {'node_a': node_a, 'node_b': node_b,
            'cond_set': list(cond_set), 'correlation': r,
            'p_value': p, 'violated': lambda max_r: p < p_threshold and abs(r) > max_r ** 1.5}


def get_dsep_implications(edges, all_nodes):
    """Find testable conditional independence implications from DAG."""
    G = nx.DiGraph(edges)
    implications = []
    for a, b in combinations(all_nodes, 2):
        if G.has_edge(a, b) or G.has_edge(b, a):
            continue
        parents = set(G.predecessors(a)) | set(G.predecessors(b))
        parents -= {a, b}
        if nx.is_d_separator(G, {a}, {b}, parents):
            implications.append((a, b, parents))
        if nx.is_d_separator(G, {a}, {b}, set()):
            implications.append((a, b, set()))
    seen = set()
    unique = []
    for a, b, s in implications:
        key = (min(a, b), max(a, b), tuple(sorted(s)))
        if key not in seen:
            seen.add(key)
            unique.append((a, b, s))
    return unique


def run_dsep_tests(data, edges, all_nodes):
    """Run d-separation tests and return violations/confirmed with adaptive threshold."""
    implications = get_dsep_implications(edges, all_nodes)
    print(f"Testable implications: {len(implications)}\n")
    buffer = {}
    for node_a, node_b, cond_set in implications:
        result = test_conditional_independence(data, node_a, node_b, cond_set)
        buffer[(node_a, node_b, frozenset(cond_set))] = result
    max_r = max(abs(r['correlation']) for r in buffer.values()) if buffer else 0
    violations, confirmed = [], []
    for (node_a, node_b, cond_set), result in buffer.items():
        if result['violated'](max_r):
            violations.append(result)
            print(f"  ✗ VIOLATED: {node_a} ⊥ {node_b} | {list(cond_set)}")
            print(f"    r={result['correlation']:.4f}, p={result['p_value']:.2e}")
        else:
            confirmed.append(result)
            print(f"  ✓ confirmed: {node_a} ⊥ {node_b} | {list(cond_set)}")
            print(f"    r={result['correlation']:.4f}, p={result['p_value']:.3f}")
    print(f"\nViolations: {len(violations)}, Confirmed: {len(confirmed)}")
    return violations, confirmed


def add_fallback_edges(current_edges, violations):
    """Add directed fallback edges for unresolved d-sep violations."""
    added = []
    edges = list(current_edges)
    for v in violations:
        a, b = v['node_a'], v['node_b']
        if (a, b) in edges or (b, a) in edges:
            continue
        G_temp = nx.DiGraph(edges)
        a_depth = len(nx.ancestors(G_temp, a)) if a in G_temp else 0
        b_depth = len(nx.ancestors(G_temp, b)) if b in G_temp else 0
        edge = (a, b) if a_depth <= b_depth else (b, a)
        G_test = nx.DiGraph(edges + [edge])
        if nx.is_directed_acyclic_graph(G_test):
            edges.append(edge)
            added.append(edge)
            print(f"  Added fallback: {edge[0]} → {edge[1]} (r={v['correlation']:.4f})")
        else:
            print(f"  Skipped (cycle): {edge}")
    if not added and not violations:
        print("All violations resolved — no fallback edges needed.")
    print(f"\nFinal: {len(edges)} edges")
    return edges, added


def run_dml_quick(data, treatment, outcome, dag_str):
    """Quick DML estimate without CI (for sensitivity/refutation)."""
    try:
        m = dowhy.CausalModel(data=data, treatment=treatment,
                              outcome=outcome, graph=dag_str)
        ident = m.identify_effect(proceed_when_unidentifiable=False)
        est = m.estimate_effect(ident, method_name="backdoor.econml.dml.DML",
            method_params={"init_params": {
                "model_y": LGBMRegressor(**LGBM_PARAMS),
                "model_t": LGBMRegressor(**LGBM_PARAMS),
                "model_final": LinearRegression(),
                "discrete_treatment": False}, "fit_params": {}})
        return est.value
    except Exception as e:
        print(f"  Failed: {e}")
        return None

## Data Loading

In [4]:
data = pd.read_csv(DATA_PATH, parse_dates=[DATE_COLUMN])
data = data.drop(columns=[c for c in STRIP_COLUMNS if c in data.columns], errors='ignore')

# Derived columns
for col_name, (source, transform) in DERIVED_COLUMNS.items():
    if transform == 'category_codes':
        data[col_name] = data[source].astype('category').cat.codes
    elif transform == 'month':
        data[col_name] = data[source].dt.month
    else:
        raise ValueError(f"Unknown transform: {transform}")

# Enrichment join
enrichment_table = None
if ENRICHMENT is not None:
    enrichment_table = pd.read_csv(
        ENRICHMENT['table_path'],
        parse_dates=ENRICHMENT.get('table_date_columns', []))
    gen_map = {}
    for _, row in enrichment_table.iterrows():
        gen_map[row[ENRICHMENT['right_key']]] = row[ENRICHMENT['value_column']]
        for extra_key, extra_val in ENRICHMENT.get('extra_mappings', []):
            gen_map[row[extra_key]] = row[extra_val]
    target = ENRICHMENT['target_name']
    data[target] = data[ENRICHMENT['join_column']].map(gen_map).fillna(
        ENRICHMENT['default_value']).astype(int)
    print(f"Enrichment '{target}': {data[target].value_counts().sort_index().to_dict()}")

print(f"Rows: {len(data):,}")
print(f"Outcome rate: {data[OUTCOME].mean():.2%}")
print(f"Date range: {data[DATE_COLUMN].min()} to {data[DATE_COLUMN].max()}")

Enrichment 'vehicle_generation': {1: 215800, 2: 242139, 3: 82081}
Rows: 540,020
Outcome rate: 4.00%
Date range: 2015-01-01 00:00:00 to 2024-08-05 00:00:00


## Step 1: D-Separation Refinement

In [5]:
# Initial d-sep on broad DAG
all_dag_nodes = list(set(n for e in DAG_BROAD_EDGES for n in e))
violations, confirmed = run_dsep_tests(data, DAG_BROAD_EDGES, all_dag_nodes)

Testable implications: 14

  ✗ VIOLATED: ambient_temp_at_dispatch_c ⊥ vehicle_reefer_kw_rated | ['month', 'hub_id_enc']
    r=0.0574, p=0.00e+00
  ✗ VIOLATED: ambient_temp_at_dispatch_c ⊥ vehicle_reefer_kw_rated | []
    r=0.0566, p=0.00e+00
  ✓ confirmed: ambient_temp_at_dispatch_c ⊥ ins_type_enc | ['month', 'hub_id_enc']
    r=0.0099, p=0.000
  ✓ confirmed: ambient_temp_at_dispatch_c ⊥ ins_type_enc | []
    r=-0.0111, p=0.000
  ✗ VIOLATED: vehicle_reefer_kw_rated ⊥ container_age_months | ['month', 'ambient_temp_at_dispatch_c', 'hub_id_enc']
    r=0.1289, p=0.00e+00
  ✗ VIOLATED: vehicle_reefer_kw_rated ⊥ container_age_months | []
    r=0.1255, p=0.00e+00
  ✗ VIOLATED: vehicle_reefer_kw_rated ⊥ hub_id_enc | []
    r=0.0478, p=2.32e-270
  ✗ VIOLATED: vehicle_reefer_kw_rated ⊥ month | []
    r=-0.0593, p=0.00e+00
  ✓ confirmed: vehicle_reefer_kw_rated ⊥ ins_type_enc | []
    r=0.0262, p=0.000
  ✓ confirmed: container_age_months ⊥ ins_type_enc | ['month', 'ambient_temp_at_dispatch_c', 'h

In [6]:
# Add enrichment variable edges and re-test
if ENRICHMENT is not None and ENRICHMENT.get('dag_edges'):
    dag_v2 = list(DAG_BROAD_EDGES) + ENRICHMENT['dag_edges']
    nodes_v2 = list(set(n for e in dag_v2 for n in e))
    print(f"Updated DAG: {len(nodes_v2)} nodes, {len(dag_v2)} edges")
    for e in ENRICHMENT['dag_edges']:
        print(f"  + {e[0]} → {e[1]}")
    print()
    violations_v2, confirmed_v2 = run_dsep_tests(data, dag_v2, nodes_v2)
    print(f"\nBefore: {len(violations)} violations → After: {len(violations_v2)} violations")
else:
    dag_v2 = list(DAG_BROAD_EDGES)
    violations_v2 = violations

Updated DAG: 8 nodes, 14 edges
  + vehicle_generation → vehicle_reefer_kw_rated
  + vehicle_generation → container_age_months
  + hub_id_enc → vehicle_generation

Testable implications: 20

  ✗ VIOLATED: ambient_temp_at_dispatch_c ⊥ vehicle_reefer_kw_rated | ['month', 'vehicle_generation', 'hub_id_enc']
    r=0.0481, p=1.06e-273
  ✓ confirmed: ambient_temp_at_dispatch_c ⊥ vehicle_generation | ['month', 'hub_id_enc']
    r=0.0371, p=0.000
  ✓ confirmed: ambient_temp_at_dispatch_c ⊥ ins_type_enc | ['month', 'hub_id_enc']
    r=0.0099, p=0.000
  ✓ confirmed: ambient_temp_at_dispatch_c ⊥ ins_type_enc | []
    r=-0.0111, p=0.000
  ✓ confirmed: excursion_flag ⊥ vehicle_generation | ['vehicle_reefer_kw_rated', 'ambient_temp_at_dispatch_c', 'container_age_months', 'hub_id_enc', 'month', 'ins_type_enc']
    r=0.0184, p=0.000
  ✗ VIOLATED: vehicle_reefer_kw_rated ⊥ container_age_months | ['month', 'ambient_temp_at_dispatch_c', 'vehicle_generation', 'hub_id_enc']
    r=0.1165, p=0.00e+00
  ✓ conf

In [7]:
# Add fallback edges for remaining violations
refined_edges, fallback_edges = add_fallback_edges(dag_v2, violations_v2)
DAG_REFINED = edges_to_dot(refined_edges)
print(DAG_REFINED)

  Added fallback: ambient_temp_at_dispatch_c → vehicle_reefer_kw_rated (r=0.0481)
  Added fallback: vehicle_reefer_kw_rated → container_age_months (r=0.1165)
  Added fallback: month → vehicle_reefer_kw_rated (r=-0.0593)
  Added fallback: ins_type_enc → vehicle_reefer_kw_rated (r=0.0690)
  Added fallback: month → vehicle_generation (r=-0.0517)
  Added fallback: hub_id_enc → ins_type_enc (r=-0.0546)

Final: 20 edges
digraph {
    container_age_months -> excursion_flag;
    ins_type_enc -> excursion_flag;
    ambient_temp_at_dispatch_c -> excursion_flag;
    hub_id_enc -> container_age_months;
    hub_id_enc -> excursion_flag;
    hub_id_enc -> ambient_temp_at_dispatch_c;
    vehicle_reefer_kw_rated -> excursion_flag;
    month -> excursion_flag;
    month -> container_age_months;
    month -> ambient_temp_at_dispatch_c;
    ambient_temp_at_dispatch_c -> container_age_months;
    vehicle_generation -> vehicle_reefer_kw_rated;
    vehicle_generation -> container_age_months;
    hub_id_enc 

## Step 2: DoWhy Identification

In [8]:
model = dowhy.CausalModel(data=data, treatment=TREATMENT,
                           outcome=OUTCOME, graph=DAG_REFINED)
identified = model.identify_effect(proceed_when_unidentifiable=False)
CONFOUNDERS = list(identified.get_backdoor_variables())
print(f"Backdoor variables: {CONFOUNDERS}")

Backdoor variables: ['ambient_temp_at_dispatch_c', 'vehicle_reefer_kw_rated', 'hub_id_enc', 'month', 'ins_type_enc']


## Step 3: DML Estimation (Continuous + Threshold)

In [9]:
BASELINE_RATE = data[OUTCOME].mean()

DML_PARAMS = {
    "init_params": {
        "model_y": LGBMRegressor(**LGBM_PARAMS),
        "model_t": LGBMRegressor(**LGBM_PARAMS),
        "model_final": LinearRegression(),
        "discrete_treatment": False,
    },
    "fit_params": {
        "inference": BootstrapInference(n_bootstrap_samples=BOOTSTRAP_SAMPLES, n_jobs=-1),
    }
}

# --- Continuous ---
estimate = model.estimate_effect(identified,
    method_name="backdoor.econml.dml.DML", method_params=DML_PARAMS)
effect_value = estimate.value
econml_obj = estimate.params['_estimator_object']
ci_raw = econml_obj.effect_interval(alpha=CI_ALPHA)
effect_ci = (ci_raw[0].flatten()[0], ci_raw[1].flatten()[0])

print(f"Continuous: {effect_value*100:.4f}pp/unit")
print(f"CI: [{effect_ci[0]*100:.4f}, {effect_ci[1]*100:.4f}]")

# --- Binary threshold (if breakpoint detected) ---
bin_value, bin_ci = None, None
if THRESHOLD_BREAKPOINT is not None:
    bin_col = f'{TREATMENT}_above_threshold'
    data[bin_col] = (data[TREATMENT] > THRESHOLD_BREAKPOINT).astype(int)
    model_bin = dowhy.CausalModel(data=data, treatment=bin_col,
        outcome=OUTCOME, graph=DAG_REFINED.replace(TREATMENT, bin_col))
    id_bin = model_bin.identify_effect(proceed_when_unidentifiable=False)
    bin_params = {"init_params": dict(DML_PARAMS["init_params"]),
                  "fit_params": dict(DML_PARAMS["fit_params"])}
    bin_params["init_params"]["discrete_treatment"] = True
    bin_est = model_bin.estimate_effect(id_bin,
        method_name="backdoor.econml.dml.DML", method_params=bin_params)
    bin_value = bin_est.value
    bin_econml = bin_est.params['_estimator_object']
    bin_ci_raw = bin_econml.effect_interval(alpha=CI_ALPHA)
    bin_ci = (bin_ci_raw[0].flatten()[0], bin_ci_raw[1].flatten()[0])
    print(f"\nThreshold (>{THRESHOLD_BREAKPOINT}): {bin_value*100:.4f}pp jump")
    print(f"CI: [{bin_ci[0]*100:.4f}, {bin_ci[1]*100:.4f}]")

Continuous: 0.0228pp/unit
CI: [0.0189, 0.0255]

Threshold (>30): 0.4757pp jump
CI: [0.3834, 0.5565]


## Step 4: Mediation Decomposition

In [10]:
G_dag = nx.DiGraph(refined_edges)

# Find mediators on directed paths
mediator_nodes = set()
for path in nx.all_simple_paths(G_dag, TREATMENT, OUTCOME):
    for node in path[1:-1]:
        mediator_nodes.add(node)

mediation_results = []
print("Candidate mediators:\n")
for m in sorted(mediator_nodes):
    parents = list(G_dag.predecessors(m))
    other_parents = [p for p in parents if p != TREATMENT]
    actionable = len(other_parents) > 0
    print(f"  {m}: parents={parents}, actionable={'YES' if actionable else 'NO'}")

if mediator_nodes:
    for med_var in mediator_nodes:
        W_med = data[CONFOUNDERS + [med_var]].values
        dml_med = LinearDML(
            model_y=LGBMRegressor(**LGBM_PARAMS),
            model_t=LGBMRegressor(**LGBM_PARAMS),
            discrete_treatment=False)
        dml_med.fit(data[OUTCOME].values, data[TREATMENT].values, W=W_med)
        direct = dml_med.effect().mean()
        mediated = effect_value - direct
        frac = mediated / effect_value if effect_value != 0 else 0
        mediation_results.append({'mediator': med_var, 'direct': direct,
                                   'mediated': mediated, 'fraction': frac})
        print(f"\n  {med_var}: direct={direct*100:.4f}, mediated={mediated*100:.4f} ({frac:.0%})")
    print(f"\n⚠ Mediation is INFORMATIONAL — assumes no mediator-outcome confounding.")
else:
    print("\nNo mediators found on directed paths. GRF is primary decomposition tool.")

Candidate mediators:


No mediators found on directed paths. GRF is primary decomposition tool.


## Step 5: GRF Heterogeneity

In [11]:
Y_grf = data[OUTCOME].values
T_grf = data[TREATMENT].values
W_grf = data[CONFOUNDERS].values
X_grf = data[GRF_EFFECT_MODIFIERS].values

grf = CausalForestDML(
    model_y=LGBMRegressor(**LGBM_PARAMS),
    model_t=LGBMRegressor(**LGBM_PARAMS),
    n_estimators=GRF_ESTIMATORS, min_samples_leaf=GRF_MIN_LEAF,
    random_state=RANDOM_STATE)
grf.fit(Y=Y_grf, T=T_grf, X=X_grf, W=W_grf)

cates = grf.effect(X=X_grf)
data['_cate'] = cates

# Slice by configured groupings
discovered_slopes = {}
for label, (col, method) in GRF_SLICES.items():
    print(f"\nCATEs by {label}:\n")
    if method == 'unique':
        for val in sorted(data[col].unique()):
            subset = data[data[col] == val]['_cate']
            discovered_slopes[val] = subset.mean()
            print(f"  {val}: {subset.mean()*100:.4f}pp/unit "
                  f"(std: {subset.std()*100:.4f})")
    elif method == 'quartile':
        q_col = f'_q_{col}'
        data[q_col] = pd.qcut(data[col], 4, duplicates='drop')
        for q, grp in data.groupby(q_col)['_cate']:
            print(f"  {q}: {grp.mean()*100:.4f}pp/unit")

# Feature importances
print("\nGRF heterogeneity drivers:")
for name, imp in sorted(zip(GRF_EFFECT_MODIFIERS, grf.feature_importances_),
                         key=lambda x: -x[1]):
    print(f"  {name}: {imp:.3f}")


CATEs by insulation type:

  PIR_foam: 0.0446pp/unit (std: 0.1385)
  PUR_foam: 0.0203pp/unit (std: 0.0708)
  VIP_panel: 0.0115pp/unit (std: 0.0643)
  XPS_foam: 0.0213pp/unit (std: 0.0658)

CATEs by reefer kw:

  (3.499, 4.3]: 0.0333pp/unit
  (4.3, 4.5]: 0.0255pp/unit
  (4.5, 5.0]: 0.0103pp/unit
  (5.0, 7.0]: 0.0118pp/unit

CATEs by ambient temp:

  (-30.301000000000002, 10.1]: -0.0002pp/unit
  (10.1, 18.0]: -0.0000pp/unit
  (18.0, 25.6]: 0.0119pp/unit
  (25.6, 52.7]: 0.0795pp/unit

GRF heterogeneity drivers:
  ambient_temp_at_dispatch_c: 0.774
  ins_type_enc: 0.123
  vehicle_reefer_kw_rated: 0.103


## Step 6: Quality Gates

In [12]:
# Nuisance R²
outcome_r2 = np.mean(cross_val_score(
    LGBMRegressor(**LGBM_PARAMS), data[CONFOUNDERS], data[OUTCOME], cv=5, scoring='r2'))
treatment_r2 = np.mean(cross_val_score(
    LGBMRegressor(**LGBM_PARAMS), data[CONFOUNDERS], data[TREATMENT], cv=5, scoring='r2'))

print(f"Outcome R²: {outcome_r2:.3f} "
      f"{'*** ABORT ***' if outcome_r2 < R2_ABORT else '⚠ weak' if outcome_r2 < R2_FLAG else '✓'}")
print(f"Treatment R²: {treatment_r2:.3f} "
      f"{'ℹ near-exogenous' if treatment_r2 < 0.05 else '✓'}")

# Sanity
iqr = data[TREATMENT].quantile(0.75) - data[TREATMENT].quantile(0.25)
implied = abs(effect_value * iqr)
max_plausible = TYPICAL_EFFECT_MAX * MAGNITUDE_MULTIPLIER
print(f"\nImplied effect over IQR ({iqr:.0f}): {implied*100:.2f}pp "
      f"(max plausible: {max_plausible*100:.0f}pp)")
print(f"Direction: {'✓' if np.sign(effect_value) == EXPECTED_DIRECTION else '⚠ wrong'}")
print(f"Magnitude: {'*** ABORT ***' if implied > max_plausible else '✓ plausible'}")

Outcome R²: 0.157 ✓
Treatment R²: -0.542 ℹ near-exogenous

Implied effect over IQR (25): 0.56pp (max plausible: 15pp)
Direction: ✓
Magnitude: ✓ plausible


## Step 7: Refutations

In [13]:
REFUTE_PARAMS = {"init_params": {
    "model_y": LGBMRegressor(**LGBM_PARAMS),
    "model_t": LGBMRegressor(**LGBM_PARAMS),
    "model_final": LinearRegression(),
    "discrete_treatment": False}, "fit_params": {}}

est_light = model.estimate_effect(identified,
    method_name="backdoor.econml.dml.DML", method_params=REFUTE_PARAMS)

placebo = model.refute_estimate(identified, est_light,
    method_name="placebo_treatment_refuter", placebo_type="permute")
random_cc = model.refute_estimate(identified, est_light,
    method_name="random_common_cause")
subset = model.refute_estimate(identified, est_light,
    method_name="data_subset_refuter", subset_fraction=0.8)

plac_eff = placebo.new_effect
cc_shift = abs(random_cc.new_effect - effect_value) / abs(effect_value) if effect_value != 0 else 0
sub_shift = abs(subset.new_effect - effect_value) / abs(effect_value) if effect_value != 0 else 0

print(f"Placebo: {plac_eff:.6f} "
      f"{'*** ABORT ***' if abs(plac_eff) > abs(effect_value)*0.5 else '✓'}")
print(f"Random CC shift: {cc_shift:.1%} "
      f"{'⚠' if cc_shift > CC_SHIFT_THRESHOLD else '✓'}")
print(f"Subset shift: {sub_shift:.1%} "
      f"{'⚠' if sub_shift > SUBSET_SHIFT_THRESHOLD else '✓'}")

Placebo: -0.000002 ✓
Random CC shift: 6.7% ✓
Subset shift: 4.6% ✓


## Step 8: E-value

In [14]:
eval_multiplier = THRESHOLD_BREAKPOINT if THRESHOLD_BREAKPOINT else iqr
abs_eff = abs(effect_value * eval_multiplier)
rr = (BASELINE_RATE + abs_eff) / BASELINE_RATE
e_point = rr + math.sqrt(rr * (rr - 1))

abs_ci = abs(effect_ci[0] * eval_multiplier)
rr_ci = max(1.0, (BASELINE_RATE + abs_ci) / BASELINE_RATE)
e_ci = rr_ci + math.sqrt(rr_ci * (rr_ci - 1)) if rr_ci > 1 else 1.0

print(f"RR at {eval_multiplier}: {rr:.2f}")
print(f"E-value (point): {e_point:.2f}")
print(f"E-value (CI): {e_ci:.2f}")

RR at 30: 1.17
E-value (point): 1.62
E-value (CI): 1.54


## Step 9: Specification Sensitivity

In [15]:
# --- Confounder drop sensitivity ---
specs_continuous = {'baseline': effect_value}
for drop_var in CONFOUNDERS:
    dag_v = edges_to_dot([(s, d) for s, d in refined_edges
                          if s != drop_var and d != drop_var])
    est = run_dml_quick(data, TREATMENT, OUTCOME, dag_v)
    if est is not None:
        specs_continuous[f'drop_{drop_var}'] = est
        print(f"drop {drop_var}: {est*100:.4f}pp (Δ: {(est-effect_value)*100:+.4f})")

cont_estimates = [v for v in specs_continuous.values() if v is not None]
cont_max_dev = (max(abs(v - effect_value) / abs(effect_value) for v in cont_estimates)
                if effect_value != 0 else 0)
print(f"\nContinuous range: [{min(cont_estimates)*100:.4f}, {max(cont_estimates)*100:.4f}]")
print(f"Max deviation: {cont_max_dev:.0%} {'⚠ spec-dependent' if cont_max_dev > 1.0 else '✓'}")

# --- Threshold sensitivity (separate unit group) ---
if THRESHOLD_BREAKPOINT is not None:
    print(f"\nThreshold sensitivity:")
    specs_threshold = {}
    for offset in THRESHOLD_OFFSETS:
        cutoff = THRESHOLD_BREAKPOINT + offset
        col = f'_thresh_{cutoff}'
        data[col] = (data[TREATMENT] > cutoff).astype(int)
        dag_v = edges_to_dot([(col if s == TREATMENT else s,
                               col if d == TREATMENT else d)
                              for s, d in refined_edges])
        est = run_dml_quick(data, col, OUTCOME, dag_v)
        if est is not None:
            specs_threshold[cutoff] = est
            print(f"  >{cutoff}: {est*100:.4f}pp")

drop ambient_temp_at_dispatch_c: 0.0255pp (Δ: +0.0028)
drop vehicle_reefer_kw_rated: 0.0127pp (Δ: -0.0101)
drop hub_id_enc: 0.0254pp (Δ: +0.0026)
drop month: 0.0204pp (Δ: -0.0023)
drop ins_type_enc: 0.0216pp (Δ: -0.0012)

Continuous range: [0.0127, 0.0255]
Max deviation: 44% ✓

Threshold sensitivity:
  >24: 0.6136pp
  >27: 0.5475pp
  >30: 0.5060pp
  >33: 0.5027pp
  >36: 0.5083pp


## Step 10: Structural Breaks + Prediction Matching

In [16]:
data['_year_month'] = data[DATE_COLUMN].dt.to_period('M')
entity_monthly = (data.groupby([ENTITY_COLUMN, '_year_month'])
    .agg(_rate=(OUTCOME, 'mean'), _mean_treatment=(TREATMENT, 'mean'),
         _n=(OUTCOME, 'count'))
    .reset_index())
entity_monthly['_ym_ts'] = entity_monthly['_year_month'].dt.to_timestamp()

known_events = None
if KNOWN_EVENTS is not None:
    known_events = pd.read_csv(KNOWN_EVENTS['table_path'],
        parse_dates=[KNOWN_EVENTS['date_column']])

matches = []
for entity in entity_monthly[ENTITY_COLUMN].unique():
    ed = entity_monthly[entity_monthly[ENTITY_COLUMN] == entity].sort_values('_year_month')
    series = ed['_rate'].values
    if len(series) < 12:
        continue
    try:
        brks = ruptures.Pelt(model="rbf").fit(series).predict(pen=3)
    except Exception:
        continue
    for bi in brks[:-1]:
        if bi < 3 or bi > len(series) - 3:
            continue
        break_date = ed.iloc[bi]['_ym_ts']
        pre = series[max(0, bi-6):bi].mean()
        post = series[bi:min(len(series), bi+6)].mean()
        actual = post - pre
        m = {'entity': entity, 'break_date': break_date,
             'actual_change': actual, 'has_known_event': False}
        if known_events is not None:
            window = pd.DateOffset(months=KNOWN_EVENTS['match_window_months'])
            ke = known_events[
                (known_events[KNOWN_EVENTS['entity_column']] == entity) &
                (known_events[KNOWN_EVENTS['date_column']] >= break_date - window) &
                (known_events[KNOWN_EVENTS['date_column']] <= break_date + window)]
            if len(ke) > 0:
                pre_t = ed.iloc[max(0, bi-3):bi]['_mean_treatment'].mean()
                post_t = ed.iloc[bi:bi+3]['_mean_treatment'].mean()
                t_delta = post_t - pre_t
                predicted = t_delta * effect_value
                m.update({'has_known_event': True, 'treatment_delta': t_delta,
                    'predicted': predicted,
                    'direction_match': np.sign(actual) == np.sign(predicted),
                    'within_ci': (effect_ci[0]*t_delta <= actual <= effect_ci[1]*t_delta
                                  if t_delta != 0 else False)})
        matches.append(m)

matched = [m for m in matches if m['has_known_event']]
ci_matches = [m for m in matched if m.get('within_ci')]
dir_matches = [m for m in matched if m.get('direction_match')]

print(f"Breaks: {len(matches)}, matched: {len(matched)}, unmatched: {len(matches)-len(matched)}")
for m in matched:
    print(f"  {m['entity']} @ {m['break_date'].strftime('%Y-%m')}: "
          f"pred={m['predicted']*100:.2f}pp actual={m['actual_change']*100:.2f}pp "
          f"dir={'✓' if m['direction_match'] else '✗'} "
          f"CI={'✓' if m['within_ci'] else '✗'}")

tier = 1 if len(ci_matches) >= 2 else (2 if len(dir_matches) >= 1 else 3)
print(f"\nTier: {tier}")

Breaks: 0, matched: 0, unmatched: 0

Tier: 3


## Step 11: Residual Diagnostics + Auto-Correction

In [17]:
# Recompute DML residuals
X_c = data[CONFOUNDERS].values
Y_v = data[OUTCOME].values
T_v = data[TREATMENT].values
y_res = np.zeros(len(data))
t_res = np.zeros(len(data))
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for tr, te in kf.split(data):
    m_y = LGBMRegressor(**LGBM_PARAMS).fit(X_c[tr], Y_v[tr])
    m_t = LGBMRegressor(**LGBM_PARAMS).fit(X_c[tr], T_v[tr])
    y_res[te] = Y_v[te] - m_y.predict(X_c[te])
    t_res[te] = T_v[te] - m_t.predict(X_c[te])
final_res = y_res - effect_value * t_res

# Autocorrelation
sorted_idx = data.sort_values(DATE_COLUMN).index
dw = durbin_watson(final_res[sorted_idx])
lb = acorr_ljungbox(final_res[sorted_idx], lags=[6, 12], return_df=True)
autocorr = dw < 1.5 or (lb['lb_pvalue'] < CI_ALPHA).any()
print(f"DW: {dw:.3f} {'⚠ autocorrelation' if autocorr else '✓'}")

DW: 1.561 ⚠ autocorrelation


In [18]:
# Unused field correlations (only DAG-internal, non-descendant fields)
treatment_desc = nx.descendants(G_dag, TREATMENT)
outcome_desc = nx.descendants(G_dag, OUTCOME)
dag_nodes = set(G_dag.nodes())
used = set(CONFOUNDERS + [TREATMENT, OUTCOME] + ID_COLUMNS
           + list(treatment_desc) + list(outcome_desc))

correlations = []
for col in data.select_dtypes(include='number').columns:
    if col in used or col.startswith('_') or col not in dag_nodes:
        continue
    vals = data[col].values
    valid = ~np.isnan(vals)
    if valid.sum() < 100:
        continue
    r, _ = pearsonr(final_res[valid], vals[valid])
    if abs(r) > 0.05:
        correlations.append({'field': col, 'correlation': r,
                            'is_meta': col in MEASUREMENT_METADATA})
correlations.sort(key=lambda x: (-x['is_meta'], -abs(x['correlation'])))

print("Residual correlations (DAG-internal only):\n")
for c in correlations:
    print(f"  {c['field']}: r={c['correlation']:.3f}"
          f"{' [MEASUREMENT]' if c['is_meta'] else ''}")
if not correlations:
    print("  None — DAG covers all relevant fields.")

Residual correlations (DAG-internal only):

  None — DAG covers all relevant fields.


In [19]:
# Auto-correction loop
corrections = []
corrected_value = effect_value
for c in correlations[:3]:
    field = c['field']
    dag_aug = edges_to_dot(refined_edges + [(field, OUTCOME)])
    corrected = run_dml_quick(data, TREATMENT, OUTCOME, dag_aug)
    if corrected is not None:
        delta_frac = (abs(corrected - corrected_value) / abs(corrected_value)
                      if corrected_value != 0 else 0)
        corrections.append({'field': field, 'is_meta': c['is_meta'],
                           'original': corrected_value, 'corrected': corrected,
                           'delta_frac': delta_frac})
        print(f"+{field}: {corrected_value*100:.4f} → {corrected*100:.4f}pp "
              f"(Δ: {delta_frac:.1%})")
        if delta_frac < 0.01:
            break
        corrected_value = corrected

print(f"\nFinal: {corrected_value*100:.4f}pp/unit")


Final: 0.0228pp/unit


## Step 12: Range Restriction

In [20]:
X_vif = data[CONFOUNDERS + [TREATMENT]].dropna()
vif = {col: variance_inflation_factor(X_vif.values, i)
       for i, col in enumerate(X_vif.columns)}
print("VIF:")
for col, v in sorted(vif.items(), key=lambda x: -x[1]):
    print(f"  {col}: {v:.1f}{' ⚠' if v > VIF_THRESHOLD else ''}")

cv = data[TREATMENT].std() / data[TREATMENT].mean()
print(f"\nCV: {cv:.3f} {'✓' if cv >= 0.10 else '⚠'}")

if THRESHOLD_BREAKPOINT is not None:
    binary = (data[TREATMENT] > THRESHOLD_BREAKPOINT).astype(int)
    ps = LogisticRegression(max_iter=1000).fit(
        data[CONFOUNDERS], binary).predict_proba(data[CONFOUNDERS])[:, 1]
    c_lo = max(np.percentile(ps[binary == 1], 5), np.percentile(ps[binary == 0], 5))
    c_hi = min(np.percentile(ps[binary == 1], 95), np.percentile(ps[binary == 0], 95))
    overlap = np.mean((ps >= c_lo) & (ps <= c_hi))
    print(f"Overlap: {overlap:.1%} {'✓' if overlap >= OVERLAP_THRESHOLD else '⚠'}")

VIF:
  vehicle_reefer_kw_rated: 11.2 ⚠
  container_age_months: 5.8
  ambient_temp_at_dispatch_c: 5.0
  month: 4.3
  hub_id_enc: 4.0
  ins_type_enc: 3.1

CV: 0.446 ✓
Overlap: 85.4% ✓


## Step 13: Externalization Test

In [21]:
ranking_match = None
if EXTERNALIZATION is not None and discovered_slopes:
    domain_ranking = EXTERNALIZATION['domain_ranking']
    discovered_ranking = sorted(discovered_slopes.keys(),
                                key=lambda k: discovered_slopes[k])

    print("Domain ranking (slowest → fastest):")
    for i, n in enumerate(domain_ranking):
        print(f"  {i+1}. {n}")
    print("\nData-discovered ranking:")
    for i, n in enumerate(discovered_ranking):
        print(f"  {i+1}. {n}: {discovered_slopes[n]*100:.4f}pp/unit")

    ranking_match = discovered_ranking == domain_ranking
    print(f"\nRanking match: {'✓ CONFIRMED' if ranking_match else '✗ DIVERGENT'}")

    if len(discovered_slopes) >= 2:
        slowest, fastest = discovered_ranking[0], discovered_ranking[-1]
        if discovered_slopes[slowest] != 0:
            data_ratio = discovered_slopes[fastest] / discovered_slopes[slowest]
            domain_ratio = EXTERNALIZATION['domain_ratio']
            ratio_ok = 0.3 < (data_ratio / domain_ratio) < 3.0
            print(f"\nSlope ratio ({fastest}/{slowest}):")
            print(f"  Data: {data_ratio:.1f}×, Domain: {domain_ratio:.1f}×")
            print(f"  Consistent: {'✓' if ratio_ok else '✗'}")

    # Allocation bias check
    alloc_col = EXTERNALIZATION.get('allocation_check_column')
    rank_col = EXTERNALIZATION['ranking_variable']
    if alloc_col and alloc_col in data.columns:
        print(f"\nAllocation ({rank_col} × {alloc_col}):")
        alloc = (data.groupby([rank_col, alloc_col])[OUTCOME].count()
                 .unstack().apply(lambda x: x / x.sum(), axis=1).round(3))
        print(alloc)
else:
    print("Externalization test: skipped (no config or no discovered slopes)")

Domain ranking (slowest → fastest):
  1. VIP_panel
  2. PUR_foam
  3. PIR_foam
  4. XPS_foam

Data-discovered ranking:
  1. VIP_panel: 0.0115pp/unit
  2. PUR_foam: 0.0203pp/unit
  3. XPS_foam: 0.0213pp/unit
  4. PIR_foam: 0.0446pp/unit

Ranking match: ✗ DIVERGENT

Slope ratio (PIR_foam/VIP_panel):
  Data: 3.9×, Domain: 4.0×
  Consistent: ✓

Allocation (container_insulation_type × hub_id):
hub_id                     HUB_MT  HUB_MW  HUB_NE  HUB_PNW  HUB_SC  HUB_SE  \
container_insulation_type                                                    
PIR_foam                    0.112   0.115   0.111    0.148   0.160   0.146   
PUR_foam                    0.079   0.114   0.234    0.089   0.130   0.190   
VIP_panel                   0.157   0.077   0.226    0.166   0.153   0.170   
XPS_foam                    0.102   0.189   0.115    0.091   0.185   0.129   

hub_id                     HUB_SW  
container_insulation_type          
PIR_foam                    0.208  
PUR_foam                    0.1

## Summary

In [22]:
print("=" * 60)
print("CAUSAL VERIFICATION RESULT")
print("=" * 60)

print(f"\nHypothesis: {TREATMENT} → {OUTCOME}")
print(f"Tier: {tier}")

print(f"\n--- DAG Refinement ---")
print(f"Broad: {len(DAG_BROAD_EDGES)} edges, Refined: {len(refined_edges)} edges")
print(f"Fallback edges: {len(fallback_edges)}")
print(f"Backdoor set: {CONFOUNDERS}")

print(f"\n--- Estimates ---")
print(f"Total: {corrected_value*100:.4f}pp/unit (CI: [{effect_ci[0]*100:.4f}, {effect_ci[1]*100:.4f}])")
if bin_value is not None:
    print(f"Threshold (>{THRESHOLD_BREAKPOINT}): {bin_value*100:.4f}pp jump "
          f"(CI: [{bin_ci[0]*100:.4f}, {bin_ci[1]*100:.4f}])")

if mediation_results:
    print(f"\n--- Mediation (INFORMATIONAL) ---")
    for mr in mediation_results:
        print(f"  via {mr['mediator']}: {mr['mediated']*100:.4f}pp ({mr['fraction']:.0%})")

if discovered_slopes:
    print(f"\n--- CATEs ---")
    for n in sorted(discovered_slopes, key=lambda k: discovered_slopes[k]):
        print(f"  {n}: {discovered_slopes[n]*100:.4f}pp/unit")

print(f"\n--- Sensitivity ---")
print(f"E-value: {e_point:.2f} (CI: {e_ci:.2f})")
print(f"Spec deviation (continuous): {cont_max_dev:.0%}")
print(f"Nuisance R²: {outcome_r2:.3f}")

print(f"\n--- Reliability ---")
print(f"VIF: {vif[TREATMENT]:.1f}, CV: {cv:.3f}")
if THRESHOLD_BREAKPOINT is not None:
    print(f"Overlap: {overlap:.1%}")

print(f"\n--- Prediction Matching ---")
print(f"Matched: {len(matched)}, direction: {len(dir_matches)}, CI: {len(ci_matches)}")

print(f"\n--- Corrections ---")
for c in corrections:
    print(f"  +{c['field']}{' [meas]' if c['is_meta'] else ''}: Δ={c['delta_frac']:.1%}")
if not corrections:
    print("  None")

print(f"\n--- Refutations ---")
print(f"Placebo: {'PASS' if abs(plac_eff) < abs(effect_value)*0.5 else 'FAIL'}")
print(f"CC: {cc_shift:.1%}, Subset: {sub_shift:.1%}")

print(f"\n--- Externalization ---")
if ranking_match is not None:
    print(f"Ranking match: {'✓' if ranking_match else '✗'}")
else:
    print("Skipped")

CAUSAL VERIFICATION RESULT

Hypothesis: container_age_months → excursion_flag
Tier: 3

--- DAG Refinement ---
Broad: 11 edges, Refined: 20 edges
Fallback edges: 6
Backdoor set: ['ambient_temp_at_dispatch_c', 'vehicle_reefer_kw_rated', 'hub_id_enc', 'month', 'ins_type_enc']

--- Estimates ---
Total: 0.0228pp/unit (CI: [0.0189, 0.0255])
Threshold (>30): 0.4757pp jump (CI: [0.3834, 0.5565])

--- CATEs ---
  VIP_panel: 0.0115pp/unit
  PUR_foam: 0.0203pp/unit
  XPS_foam: 0.0213pp/unit
  PIR_foam: 0.0446pp/unit

--- Sensitivity ---
E-value: 1.62 (CI: 1.54)
Spec deviation (continuous): 44%
Nuisance R²: 0.157

--- Reliability ---
VIF: 5.8, CV: 0.446
Overlap: 85.4%

--- Prediction Matching ---
Matched: 0, direction: 0, CI: 0

--- Corrections ---
  None

--- Refutations ---
Placebo: PASS
CC: 6.7%, Subset: 4.6%

--- Externalization ---
Ranking match: ✗
